In [6]:
import pandas as pd
import ast
from sklearn.model_selection import GroupKFold
import numpy as np
import os
import re

In [7]:
df = pd.read_csv('./data/annotated/full_ner/all_annotations_minimal_fixed_multi.csv')

In [8]:
# Replace with your actual folder path
folder_path = "./data/annotated"

excluded_ids = []

# Loop through all .txt files in the folder
for filename in os.listdir(folder_path):
    if filename.endswith(".txt"):
        file_path = os.path.join(folder_path, filename)
        with open(file_path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                excluded_ids.append(line)

# Optional: make sure IDs are unique
excluded_ids = list(set(excluded_ids))

print("Excluded IDs collected from all files:")
print(len(excluded_ids))

Excluded IDs collected from all files:
87


In [12]:
# First, extract the base ID (e.g., 'My_pdf931' from 'My_pdf931_title^abstract')
df['base_doc_id'] = df['doc_id'].apply(lambda x: x.split('_')[0] + '_' + x.split('_')[1] if '_' in x else x)

# Then filter out rows where base_doc_id is in excluded_ids
filtered_df = df[~df['base_doc_id'].isin(excluded_ids)].reset_index(drop=True)

print(f"Filtered DataFrame: {len(df)} → {len(filtered_df)} rows")

Filtered DataFrame: 1614 → 1440 rows


In [14]:
df = filtered_df.copy()

In [16]:
len(set(df['doc_id']))

1440

In [18]:
df['tokens'] = df['tokens'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
df['ner_tags'] = df['ner_tags'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
df['annotations_tuples'] = df['annotations_tuples'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

df['tokens_length'] = df['tokens'].apply(len)
df['ner_tags_length'] = df['ner_tags'].apply(len)
df['length_match'] = df['tokens_length'] == df['ner_tags_length']
df.head()

,doc_id,Text,tokens,ner_tags,annotations_tuples,base_doc_id,tokens_length,ner_tags_length,length_match
0,My_pdf931_title^abstract,Pioglitazone attenuates mitochondrial dysfunct...,"[Pioglitazone, attenuates, mitochondrial, dysf...","[B-therapy-drug, O, O, O, O, O, O, O, O, O, O,...","[(0, 12, therapy-drug, Pioglitazone), (122, 14...",My_pdf931,396,396,True
1,My_pdf931_method,\r\n6 Materials and Methods\r\n6.1 Controlled ...,"[\r\n, 6, Materials, and, Methods, \r\n, 6.1, ...","[O, O, O, O, O, O, O, O, O, O, O, O, O, B-welf...","[(72, 169, welfare, Studies were approved by t...",My_pdf931,1453,1453,True
2,My_pdf69new_title^abstract,Inhibition of Factor XIa Reduces the Frequency...,"[Inhibition, of, Factor, XIa, Reduces, the, Fr...","[O, O, O, O, O, O, O, O, O, O, O, O, O, B-dise...","[(93, 120, disease, Carotid Arterial Thrombosi...",My_pdf69new,336,336,True
3,My_pdf69new_method,\r\nMaterials and Methods\r\nAnimals. All anim...,"[\r\n, Materials, and, Methods, \r\n, Animals,...","[O, O, O, O, O, O, O, B-age|B-weight, I-age|I-...","[(34, 219, age, All animal studies were perfor...",My_pdf69new,1996,1996,True
4,My_pdf764_title^abstract,Resveratrol inhibits paclitaxel-induced neurop...,"[Resveratrol, inhibits, paclitaxel, -, induced...","[B-therapy-drug, O, O, O, O, B-disease, I-dise...","[(0, 11, therapy-drug, Resveratrol), (40, 56, ...",My_pdf764,456,456,True


## Filter for Drug and Disease

In [20]:
# Filter NER tags, handling compound tags like 'B-disease|B-age'
def filter_ner_tags(tag_list, allowed_tags):
    filtered = []
    for tag in tag_list:
        parts = tag.split('|')
        # Keep only allowed tags and remove duplicates while preserving order
        seen = set()
        kept = [p for p in parts if p in allowed_tags and not (p in seen or seen.add(p))]

        # Remove conflicting B- and I- tags for the same entity type
        entity_types = {}
        for p in kept:
            prefix, entity = p.split('-', 1)
            entity_types.setdefault(entity, set()).add(prefix)

        # Only keep tags that are not conflicting (i.e., only B or I for the same entity)
        final_tags = []
        for p in kept:
            prefix, entity = p.split('-', 1)
            if len(entity_types[entity]) == 1:  # Either only B or only I
                final_tags.append(p)
            else:
                if prefix == "B":
                    final_tags.append(p)

        filtered.append('|'.join(final_tags) if final_tags else 'O')
    return filtered

# Replace parts of a tag using a mapping dictionary
def replace_tag(tag, tag_mapping):
    for key, value in tag_mapping.items():
        if key in tag:
            tag = tag.replace(key, value)
    return tag

# Filter annotations_tuples based on tag containing certain substrings
def filter_annotations_tuples(annotations, allowed_substrings):
    return [
        annotation for annotation in annotations
        if any(sub in annotation[2] for sub in allowed_substrings)
    ]

# Replace tag in a single annotation tuple using the mapping
def replace_tag_in_tuple(annotation, tag_mapping):
    tag = annotation[2]
    new_tag = replace_tag(tag, tag_mapping)
    return (annotation[0], annotation[1], new_tag, annotation[3])

def count_b_tags(ner_tags, tag_mapping):
    counts = {}
    for base_tag in tag_mapping.values():
        b_tag = f'B-{base_tag}'
        counts[b_tag] = ner_tags.count(b_tag)
    return counts

In [23]:
tag_list = ['B-STRAIN|I-STRAIN', 'I-STRAIN|B-AGE', 'B-DISEASE|B-DISEASE', 'O']
allowed_tags = ['B-STRAIN', 'I-STRAIN', 'B-AGE', 'B-DISEASE']

print(filter_ner_tags(tag_list, allowed_tags))

['B-STRAIN', 'I-STRAIN|B-AGE', 'B-DISEASE', 'O']


In [25]:
def process_ner_dataframe(df, allowed_tags, tag_mapping):
    # Step 1: Ensure annotations_tuples is parsed
    df = df.copy()
    df['annotations_tuples'] = df['annotations_tuples'].apply(
        lambda x: eval(x) if isinstance(x, str) else x
    )

    # Step 2: Filter ner_tags based on allowed_tags
    df['ner_tags'] = df['ner_tags'].apply(
        lambda tags: filter_ner_tags(tags, allowed_tags)
    )

    # Step 3: Filter annotations based on substrings in allowed tags
    allowed_substrings = [tag.split('-')[-1] for tag in allowed_tags]
    df['annotations_tuples'] = df['annotations_tuples'].apply(
        lambda annots: filter_annotations_tuples(annots, allowed_substrings)
    )

    # Step 4: Replace tags using the mapping
    df['ner_tags'] = df['ner_tags'].apply(
        lambda tags: [replace_tag(tag, tag_mapping) for tag in tags]
    )
    df['annotations_tuples'] = df['annotations_tuples'].apply(
        lambda tags: [replace_tag_in_tuple(tag, tag_mapping) for tag in tags]
    )

    # Step 5: Count B-TAGs
    count_df = df['ner_tags'].apply(lambda tags: count_b_tags(tags, tag_mapping))
    count_df = pd.DataFrame(count_df.tolist())
    df = pd.concat([df, count_df], axis=1)

    return df

In [664]:
# --- Configuration ---
df_disease_drug = df.copy()

# Tags you want to keep (exact matches, including BIO prefix)
allowed_tags = ['B-therapy-drug', 'I-therapy-drug', 'B-disease', 'I-disease']

# Mapping for replacing tag components
tag_mapping = {
    'therapy-drug': 'DRUG',
    'disease': 'DISEASE'
}

# --- Apply to DataFrame ---

df_disease_drug = process_ner_dataframe(df_disease_drug, allowed_tags, tag_mapping)

print(df_disease_drug.shape)
df_disease_drug.head()

(1440, 11)


,doc_id,Text,tokens,ner_tags,annotations_tuples,base_doc_id,tokens_length,ner_tags_length,length_match,B-DRUG,B-DISEASE
0,My_pdf931_title^abstract,Pioglitazone attenuates mitochondrial dysfunct...,"[Pioglitazone, attenuates, mitochondrial, dysf...","[B-DRUG, O, O, O, O, O, O, O, O, O, O, O, O, O...","[(0, 12, DRUG, Pioglitazone), (122, 144, DISEA...",My_pdf931,396,396,True,15,6
1,My_pdf931_method,\r\n6 Materials and Methods\r\n6.1 Controlled ...,"[\r\n, 6, Materials, and, Methods, \r\n, 6.1, ...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...","[(1310, 1322, DRUG, Pioglitazone), (1342, 1350...",My_pdf931,1453,1453,True,10,0
2,My_pdf69new_title^abstract,Inhibition of Factor XIa Reduces the Frequency...,"[Inhibition, of, Factor, XIa, Reduces, the, Fr...","[O, O, O, O, O, O, O, O, O, O, O, O, O, B-DISE...","[(93, 120, DISEASE, Carotid Arterial Thrombosi...",My_pdf69new,336,336,True,1,4
3,My_pdf69new_method,\r\nMaterials and Methods\r\nAnimals. All anim...,"[\r\n, Materials, and, Methods, \r\n, Animals,...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...","[(3936, 3950, DRUG, FXIa inhibitor), (5193, 52...",My_pdf69new,1996,1996,True,2,0
4,My_pdf764_title^abstract,Resveratrol inhibits paclitaxel-induced neurop...,"[Resveratrol, inhibits, paclitaxel, -, induced...","[B-DRUG, O, O, O, O, B-DISEASE, I-DISEASE, O, ...","[(0, 11, DRUG, Resveratrol), (40, 56, DISEASE,...",My_pdf764,456,456,True,11,4


## Filter for Strain

In [667]:
# --- Configuration ---
df_strain = df.copy()

# Tags you want to keep (exact matches, including BIO prefix)
allowed_tags = ['B-strain', 'I-strain']

# Mapping for replacing tag components
tag_mapping = {
    'strain': 'STRAIN'
}

# --- Apply to DataFrame ---

df_strain = process_ner_dataframe(df_strain, allowed_tags, tag_mapping)

print(df_strain.shape)
df_strain.head()

(1440, 10)


,doc_id,Text,tokens,ner_tags,annotations_tuples,base_doc_id,tokens_length,ner_tags_length,length_match,B-STRAIN
0,My_pdf931_title^abstract,Pioglitazone attenuates mitochondrial dysfunct...,"[Pioglitazone, attenuates, mitochondrial, dysf...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...",[],My_pdf931,396,396,True,0
1,My_pdf931_method,\r\n6 Materials and Methods\r\n6.1 Controlled ...,"[\r\n, 6, Materials, and, Methods, \r\n, 6.1, ...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...","[(182, 196, STRAIN, Sprague Dawley)]",My_pdf931,1453,1453,True,1
2,My_pdf69new_title^abstract,Inhibition of Factor XIa Reduces the Frequency...,"[Inhibition, of, Factor, XIa, Reduces, the, Fr...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...","[(607, 624, STRAIN, New Zealand White)]",My_pdf69new,336,336,True,1
3,My_pdf69new_method,\r\nMaterials and Methods\r\nAnimals. All anim...,"[\r\n, Materials, and, Methods, \r\n, Animals,...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, B-S...","[(76, 93, STRAIN, New Zealand White)]",My_pdf69new,1996,1996,True,1
4,My_pdf764_title^abstract,Resveratrol inhibits paclitaxel-induced neurop...,"[Resveratrol, inhibits, paclitaxel, -, induced...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...","[(457, 471, STRAIN, Sprague Dawley)]",My_pdf764,456,456,True,1


In [669]:
df_strain['B-STRAIN'].sum()

1191

## Filter for Animals Nr

In [30]:
# --- Configuration ---
df_animal_nr = df.copy()

# Tags you want to keep (exact matches, including BIO prefix)
allowed_tags = ['B-animals-number', 'I-animals-number']

# Mapping for replacing tag components
tag_mapping = {
    'animals-number': 'ANIMALS-NR'
}

# --- Apply to DataFrame ---

df_animal_nr = process_ner_dataframe(df_animal_nr, allowed_tags, tag_mapping)

print(df_animal_nr.shape)
df_animal_nr.head()

(1440, 10)


,doc_id,Text,tokens,ner_tags,annotations_tuples,base_doc_id,tokens_length,ner_tags_length,length_match,B-ANIMALS-NR
0,My_pdf931_title^abstract,Pioglitazone attenuates mitochondrial dysfunct...,"[Pioglitazone, attenuates, mitochondrial, dysf...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...",[],My_pdf931,396,396,True,0
1,My_pdf931_method,\r\n6 Materials and Methods\r\n6.1 Controlled ...,"[\r\n, 6, Materials, and, Methods, \r\n, 6.1, ...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...","[(171, 176, ANIMALS-NR, Fifty)]",My_pdf931,1453,1453,True,1
2,My_pdf69new_title^abstract,Inhibition of Factor XIa Reduces the Frequency...,"[Inhibition, of, Factor, XIa, Reduces, the, Fr...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...",[],My_pdf69new,336,336,True,0
3,My_pdf69new_method,\r\nMaterials and Methods\r\nAnimals. All anim...,"[\r\n, Materials, and, Methods, \r\n, Animals,...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...",[],My_pdf69new,1996,1996,True,0
4,My_pdf764_title^abstract,Resveratrol inhibits paclitaxel-induced neurop...,"[Resveratrol, inhibits, paclitaxel, -, induced...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...",[],My_pdf764,456,456,True,0


In [36]:
df_animal_nr[df_animal_nr['B-ANIMALS-NR']==1]

,doc_id,Text,tokens,ner_tags,annotations_tuples,base_doc_id,tokens_length,ner_tags_length,length_match,B-ANIMALS-NR
1,My_pdf931_method,\r\n6 Materials and Methods\r\n6.1 Controlled ...,"[\r\n, 6, Materials, and, Methods, \r\n, 6.1, ...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...","[(171, 176, ANIMALS-NR, Fifty)]",My_pdf931,1453,1453,True,1
11,My_pdf238new_method,\r\n2. Materials and methods\r\nExperiments we...,"[\r\n, 2, ., Materials, and, methods, \r\n, Ex...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...","[(140, 142, ANIMALS-NR, 10)]",My_pdf238new,367,367,True,1
20,My_pdf556new_title^abstract,Is selective antegrade cerebral perfusion supe...,"[Is, selective, antegrade, cerebral, perfusion...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...","[(536, 544, ANIMALS-NR, Eighteen)]",My_pdf556new,496,496,True,1
21,My_pdf556new_method,\r\nMATERIALS AND METHODS\r\nAnimal Care\r\nAl...,"[\r\n, MATERIALS, AND, METHODS, \r\n, Animal, ...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...","[(317, 325, ANIMALS-NR, Eighteen)]",My_pdf556new,1148,1148,True,1
28,My_pdf463new_title^abstract,Anticonvulsant and sleep-waking influences of ...,"[Anticonvulsant, and, sleep, -, waking, influe...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...","[(95, 98, ANIMALS-NR, Six)]",My_pdf463new,111,111,True,1
...,...,...,...,...,...,...,...,...,...,...
1415,My_pdf174new_method,\r\n2 | MATERIALS AND METHODS\r\n2.1 | Viruses...,"[\r\n, 2, |, MATERIALS, AND, METHODS, \r\n, 2....","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...","[(9720, 9725, ANIMALS-NR, Sixty)]",My_pdf174new,2458,2458,True,1
1424,My_pdf231new_title^abstract,"Therapeutic efficacy of Ac-DMQD-CHO, a caspase...","[Therapeutic, efficacy, of, Ac, -, DMQD, -, CH...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...","[(234, 240, ANIMALS-NR, Thirty)]",My_pdf231new,208,208,True,1
1425,My_pdf231new_method,\r\n2. Materials and methods\r\n2.1. Animal pr...,"[\r\n, 2, ., Materials, and, methods, \r\n, 2....","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...","[(1341, 1347, ANIMALS-NR, Thirty)]",My_pdf231new,1335,1335,True,1
1427,My_pdf876_method,\r\n2 Material and methods\r\n2.1 Animal and r...,"[\r\n, 2, Material, and, methods, \r\n, 2.1, A...","[O, O, O, O, O, O, O, O, O, O, O, B-ANIMALS-NR...","[(51, 63, ANIMALS-NR, Seventy-five)]",My_pdf876,615,615,True,1


## Cross-Valid Splits

In [40]:
import os
import json
from sklearn.model_selection import GroupKFold
import pandas as pd

def is_valid_json(obj):
    """Check if an object can be serialized to JSON."""
    try:
        json.dumps(obj)
        return True
    except (TypeError, ValueError):
        return False


def split_and_save_kfolds(
    df,
    group_column='doc_id',
    group_key_position=1,
    text_columns=['tokens', 'Text', 'ner_tags', 'doc_id'],
    n_splits=10,
    save_loc='./data/kfold_splits/',
    save_full=True,
    full_json_path='./data/kfold_splits/full_dataset.json',
    file_prefix='fold'
):
    """
    Split dataframe into K folds using GroupKFold and save train/test splits as CSV and JSON.

    Parameters:
        df (pd.DataFrame): The dataframe to split.
        group_column (str): Column name to extract group key from.
        group_key_position (int): Position to use when splitting the group_column by '_'.
        text_columns (list): List of columns to include in the JSON output.
        n_splits (int): Number of folds.
        save_loc (str): Directory to save fold files.
        save_full (bool): Whether to save the combined full dataset.
        full_json_path (str): Path to save the combined JSON file.
        file_prefix (str): Prefix for saved files, e.g., 'drug_disease'.
    """
    df = df.copy()

    # Step 1: Extract group key from the doc_id or other column
    df['group_key'] = df[group_column].apply(lambda x: x.split('_')[group_key_position])

    # Step 2: Set up GroupKFold
    group_kfold = GroupKFold(n_splits=n_splits)
    groups = df['group_key']
    X = df  # Features can be entire df or subset
    y = df['ner_tags']  # Assuming ner_tags is the target

    # Step 3: Ensure save location exists
    os.makedirs(save_loc, exist_ok=True)

    full_data = []

    for fold, (train_idx, test_idx) in enumerate(group_kfold.split(X, y, groups)):
        print(f"🔄 Fold {fold}: {len(train_idx)} train / {len(test_idx)} test")

        train_data = df.iloc[train_idx]
        test_data = df.iloc[test_idx]

        # Save CSV
        train_data.to_csv(f'{save_loc}/{file_prefix}_train_fold_{fold}.csv', index=False)
        test_data.to_csv(f'{save_loc}/{file_prefix}_test_fold_{fold}.csv', index=False)

        # Convert to JSON
        train_json = train_data[text_columns].to_dict(orient='records')
        test_json = test_data[text_columns].to_dict(orient='records')

        with open(f'{save_loc}/{file_prefix}_train_fold_{fold}.json', 'w', encoding='utf-8') as f:
            json.dump(train_json, f, indent=4)

        with open(f'{save_loc}/{file_prefix}_test_fold_{fold}.json', 'w', encoding='utf-8') as f:
            json.dump(test_json, f, indent=4)

        # Add to full dataset if valid
        full_data.extend([row for row in train_json if is_valid_json(row)])
        full_data.extend([row for row in test_json if is_valid_json(row)])

    if save_full:
        os.makedirs(os.path.dirname(full_json_path), exist_ok=True)
        with open(full_json_path, 'w', encoding='utf-8') as f:
            json.dump(full_data, f, indent=4)
        print(f"✅ Full dataset saved as JSON array: {full_json_path}")


In [617]:
split_and_save_kfolds(
    df=df_disease_drug,
    group_column='doc_id',
    group_key_position=1,
    text_columns=['tokens', 'Text', 'ner_tags', 'doc_id'],
    n_splits=10,
    save_loc='./data/finetuning_ner/drug_disease/k_fold_splits/',
    save_full=True,
    full_json_path='./data/finetuning_ner/drug_disease/full_ds_drug_disease_ner.json',
    file_prefix='drug_disease'
)

🔄 Fold 0: 1296 train / 144 test
🔄 Fold 1: 1296 train / 144 test
🔄 Fold 2: 1296 train / 144 test
🔄 Fold 3: 1296 train / 144 test
🔄 Fold 4: 1296 train / 144 test
🔄 Fold 5: 1296 train / 144 test
🔄 Fold 6: 1296 train / 144 test
🔄 Fold 7: 1296 train / 144 test
🔄 Fold 8: 1296 train / 144 test
🔄 Fold 9: 1296 train / 144 test
✅ Full dataset saved as JSON array: ./data/finetuning_ner/drug_disease/full_ds_drug_disease_ner.json


In [673]:
split_and_save_kfolds(
    df=df_strain,
    group_column='doc_id',
    group_key_position=1,
    text_columns=['tokens', 'Text', 'ner_tags', 'doc_id'],
    n_splits=10,
    save_loc='./data/finetuning_ner/strain/k_fold_splits/',
    save_full=True,
    full_json_path='./data/finetuning_ner/strain/full_ds_strain_ner.json',
    file_prefix='strain'
)

🔄 Fold 0: 1296 train / 144 test
🔄 Fold 1: 1296 train / 144 test
🔄 Fold 2: 1296 train / 144 test
🔄 Fold 3: 1296 train / 144 test
🔄 Fold 4: 1296 train / 144 test
🔄 Fold 5: 1296 train / 144 test
🔄 Fold 6: 1296 train / 144 test
🔄 Fold 7: 1296 train / 144 test
🔄 Fold 8: 1296 train / 144 test
🔄 Fold 9: 1296 train / 144 test
✅ Full dataset saved as JSON array: ./data/finetuning_ner/strain/full_ds_strain_ner.json


In [44]:
split_and_save_kfolds(
    df=df_animal_nr,
    group_column='doc_id',
    group_key_position=1,
    text_columns=['tokens', 'Text', 'ner_tags', 'doc_id'],
    n_splits=10,
    save_loc='./data/finetuning_ner/animals_nr/k_fold_splits/',
    save_full=True,
    full_json_path='./data/finetuning_ner/animals_nr/full_ds_animals_nr_ner.json',
    file_prefix='animal_nr'
)

🔄 Fold 0: 1296 train / 144 test
🔄 Fold 1: 1296 train / 144 test
🔄 Fold 2: 1296 train / 144 test
🔄 Fold 3: 1296 train / 144 test
🔄 Fold 4: 1296 train / 144 test
🔄 Fold 5: 1296 train / 144 test
🔄 Fold 6: 1296 train / 144 test
🔄 Fold 7: 1296 train / 144 test
🔄 Fold 8: 1296 train / 144 test
🔄 Fold 9: 1296 train / 144 test
✅ Full dataset saved as JSON array: ./data/finetuning_ner/animals_nr/full_ds_animals_nr_ner.json


#### validating json format

In [83]:
with open('./finetuning_ner/drug_disease/full_ds_drug_disease_ner.json', 'r', encoding='utf-8') as json_file:
    data = [json.loads(line) for line in json_file]  # Read each line as a JSON object

# Convert the list of dictionaries into a DataFrame
df_loaded = pd.DataFrame(data)

In [99]:
input_file = './finetuning_ner/drug_disease/full_ds_drug_disease_ner.json'
valid_data = []

# Read and filter out bad JSON lines
with open(input_file, 'r', encoding='utf-8') as json_file:
    for line in json_file:
        try:
            entry = json.loads(line.strip())  # Parse JSON line
            valid_data.append(entry)  # Store only valid entries
        except json.JSONDecodeError as e:
            print(f"Skipping corrupted JSON entry: {e}")  # Log error

# Convert to DataFrame
df_loaded = pd.DataFrame(valid_data)

In [111]:
output_file = './finetuning_ner/drug_disease/full_ds_drug_disease_ner_cleaned_array.json'

# Save entire dataset as a JSON array (formatted)
with open(output_file, 'w', encoding='utf-8') as json_file:
    json.dump(valid_data, json_file, indent=4)  # Pretty format with indentation

In [113]:
import json

file_path = './finetuning_ner/drug_disease/full_ds_drug_disease_ner_cleaned_array.json'

try:
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    print("JSON file is valid!")
except json.JSONDecodeError as e:
    print(f"JSON Decode Error: {e}")


JSON file is valid!
